# rift `biomass-e2e` — DPS job runner (bounding-box driven)

Discover ESA **BIOMASS Level-1A SCS** granules over an area of interest on the ESA MAAP
STAC catalogue with `pystac-client`, and submit one MAAP DPS job per granule to the
`rift-biomass-e2e` OGC process. Each job runs the full pipeline on the worker: `rift
biomass2cog` (granule → amplitude COGs), then per-pol `*_intensity.tif` COGs, then the
**nisar-crevasse** BIOMASS gate + U-Net model (→ `gate_prob.tif` + `unet_prob.tif`). The
worker resolves the Item, authenticates to ESA with a token, and streams the product zip
itself — so no pre-staging is needed.

Defaults here target a **real full-granule run**: the **`maap-dps-worker-16gb`** queue with
**`max_tiles=""`** (no cap). The U-Net is CPU-bound and BIOMASS geocoding is memory-heavy, so
a full-granule pass takes tens of minutes to hours; a heavy granule may need
**`maap-dps-worker-32gb`** (the algorithm's `ram_min` is 32 GB). For a quick plumbing check,
switch to **`maap-dps-sandbox`** (8 GB, 10-min cap) and set **`max_tiles="100"`**.

Run this in a **MAAP Hub (OGC) workspace** so maap-py v5 is available.

**Prereqs**
- The `rift-biomass-e2e` OGC process is registered (see `maap/biomass_e2e/README.md`).
- ESA credentials registered as MAAP secrets `ESA_MAAP_CLIENT_SECRET` / `ESA_OFFLINE_TOKEN`,
  and NASA Earthdata credentials `EARTHDATA_USERNAME` / `EARTHDATA_PASSWORD` (for the DEM)
  (one-time; see the README). Discovery below needs only the public STAC (no auth); the
  *worker* needs the secrets to download the granule and DEM.
- `pystac-client` installed (`pip install pystac-client`).

All four polarizations (HH,HV,VH,VV) are geocoded (the U-Net is a 4-channel model), so pols
are not a job input.

In [ ]:
# One-time in a fresh workspace:
# %pip install pystac-client
import datetime, json, os, time
import pandas as pd
from pystac_client import Client
from maap.maap import MAAP

maap = MAAP()

ESA_STAC_API_URL = "https://catalog.maap.eo.esa.int/catalogue/"
COLLECTION = "BiomassLevel1a"   # ESA BIOMASS Level-1A SCS (range-doppler SLCs)

# STAC discovery is public — no login needed here. (The DPS worker uses the ESA secrets.)
catalog = Client.open(ESA_STAC_API_URL)
print("opened", ESA_STAC_API_URL)

## 1. Area of interest

Thwaites / Pine Island sector. The bounding box `(W, S, E, N)` is set in the search cell
below (§2). Source polygon, for reference:
`POLYGON((-101.9351 -74.7848,-102.4704 -75.4215,-99.852 -75.5511,-99.4242 -74.9087,-101.9351 -74.7848))`

## 2. Search for BIOMASS L1A granules (pystac-client / ESA STAC)

BIOMASS data is discovered on the **ESA MAAP STAC** (`catalog.maap.eo.esa.int`), not CMR.
See the MAAP tutorial
[Searching NISAR/BIOMASS overlapping data](https://docs.maap-project.org/en/latest/technical_tutorials/search/searching_NISAR_BIOMASS_overlapping_data.html#2\)-ESA-BIOMASS).

- `collections=["BiomassLevel1a"]` — the L1A SCS collection
- `bbox=[W, S, E, N]` — numeric list
- `datetime="START/END"` — RFC 3339 interval

Each Item's `id` is what we submit; `assets["product"].href` is the full zip the worker pulls.

In [ ]:
# AOI bounding box (W, S, E, N) from the Thwaites/PIG polygon.
BBOX = [-102.4704, -75.5511, -99.4242, -74.7848]

# Acquisition window (adjust as needed).
START = "2026-07-08T00:00:00Z"
END   = "2026-07-31T23:59:59Z"

search = catalog.search(
    collections=[COLLECTION],
    bbox=BBOX,
    datetime=f"{START}/{END}",
    limit=200,        # page size — make it >= your expected result count
    max_items=200,
)
# one shot instead of lazy multi-page iteration (avoids ESA resto paging-token
# expiry: `.items()` follows `next` links whose short-lived cursor can 400).
items = list(search.item_collection())
print(f"{len(items)} items")
len(items)

In [ ]:
# Peek at the first item's id + product asset.
if items:
    it = items[0]
    print("id        :", it.id)
    print("datetime  :", it.datetime)
    print("bbox      :", it.bbox)
    prod = it.assets.get("product")
    print("product   :", prod.href if prod else "<missing 'product' asset>")
else:
    print("No items — widen the date window or bbox, or verify the collection id.")

## 3. Build the submit list

The worker downloads the product itself (STAC resolve + ESA token + `requests` streaming),
so we just pass the **`item_id`** per granule — no hrefs, no pre-staging. We keep the product
href only for the submission log.

In [ ]:
granules = []
for it in items:
    prod = it.assets.get("product")
    granules.append({
        "item_id": it.id,
        "datetime": str(it.datetime),
        "product_href": prod.href if prod else None,
    })

print(f"{len(granules)} granules")
for gr in granules[:5]:
    print(" ", gr["item_id"])

## 4. Resolve the deployed OGC process id

`maap.list_algorithms()` returns `{"processes": [{"processID": 89, "id": "rift-biomass-e2e", ...}]}`.
**`submit_job` needs the numeric `processID`** (e.g. `89`), *not* the string `id`
(`"rift-biomass-e2e"`) — passing the string (or a stale/deleted id) yields a 500 auth error.
We pick the newest matching `processID`.

In [ ]:
resp = maap.list_algorithms()
procs = resp.json().get("processes", []) if resp.status_code == 200 else []
for p in procs:
    print(p.get("processID"), "|", p.get("id"), "|", p.get("version"))

# submit_job wants the NUMERIC processID (not the string id). Grab the newest matching one.
matches = [p for p in procs
           if "biomass-e2e" in str(p.get("id", "")).lower()
           or "biomass-e2e" in str(p.get("title", "")).lower()]
PROCESS_ID = max((p["processID"] for p in matches), default=None)
print("\nPROCESS_ID =", PROCESS_ID)

## 5. Submit one DPS job per granule

Start with a **single** granule (`granules[:1]`) to validate the plumbing quickly. We pass
`item_id` (+ the inference knobs); the worker resolves and downloads the product with the ESA
secrets, geocodes it, derives intensity COGs, and runs the crevasse model. `submit_job`
returns HTTP 202 with a `jobID`.

The inference knobs below follow the algorithm developer's suggested BIOMASS defaults:
`GATE_THRESH="0.5"`, `EDGE_MARGIN="64"` (removes the tile-seam artifact), and the
**grounded-ice pre-filter** on — `BEDMAP_MASK="true"` (the crevasse-bundled Bedmap3 mask)
with `MIN_GROUNDED="0.80"`, which drops candidate tiles less than 80% grounded ice
(floating shelf / sea ice / ocean / rock) *before* any gate/U-Net scoring. `CROP_TO_SCANNED`
is off because it is mutually exclusive with `EDGE_MARGIN` (the wrapper prefers `edge_margin`
and warns if both are set).

> **Note on `edge_margin` + `min_grounded`.** With `EDGE_MARGIN>0`, the crevasse export
> path uses the grounded-filtered candidates only to compute the rescoring **bounding box**,
> then rescores every tile inside that box — so grounded filtering narrows the box but is not
> strictly per-tile. Set `EDGE_MARGIN="0"` if you want the grounded filter applied tile-by-tile.

Defaults run the **full granule** (`MAX_TILES=""`) on `maap-dps-worker-16gb` — expect tens of
minutes to hours, and move to `maap-dps-worker-32gb` for a heavy granule. For a quick plumbing
check instead, set `MAX_TILES="100"` and use `maap-dps-sandbox` (8 GB / 10 min); a capped run
without crop is a mostly-NaN full-granule raster, which only proves the job is accepted.

In [ ]:
QUEUE = "maap-dps-worker-16gb"   # real-run queue. Use maap-dps-sandbox (8 GB, 10-min cap)
                                  # for a quick plumbing test (pair with MAX_TILES="100"),
                                  # or maap-dps-worker-32gb for a heavy full-granule run.
TAG = "biomass-e2e-thwaites"
AMP_ONLY = "false"
MAX_TILES = ""                   # "" = full granule; set e.g. "100" to cap tiles for a quick test
CROP_TO_SCANNED = "false"        # off: edge_margin (below) is set, and the two are mutually
                                  # exclusive. A capped run without crop is a mostly-NaN
                                  # full-granule raster -- fine for a plumbing test.
GATE_THRESH = "0.5"              # gate probability threshold for flagging tiles
EDGE_MARGIN = "64"               # remove the tile-seam artifact (mutually excl. w/ crop)
BEDMAP_MASK = "true"             # "true" = crevasse-bundled Bedmap3 mask; a path = your own;
                                  # "" = disable grounded filtering
MIN_GROUNDED = "0.80"            # drop candidate tiles <80% grounded ice before scoring;
                                  # "" = off. Requires BEDMAP_MASK. NB: with EDGE_MARGIN>0
                                  # this narrows the rescoring bbox, not each tile.

SUBMIT = granules[:1]             # <-- widen to `granules` once the first job succeeds
assert PROCESS_ID, "PROCESS_ID not resolved — is the process deployed? (see §4)"
assert SUBMIT, "No granules — check the search cells."

rows = []
for i, gr in enumerate(SUBMIT, start=1):
    inputs = {
        "item_id": gr["item_id"],
        "amp_only": AMP_ONLY,
        "max_tiles": MAX_TILES,
        "crop_to_scanned": CROP_TO_SCANNED,
        "gate_thresh": GATE_THRESH,
        "edge_margin": EDGE_MARGIN,
        "bedmap_mask": BEDMAP_MASK,
        "min_grounded": MIN_GROUNDED,
    }
    r = maap.submit_job(process_id=PROCESS_ID, inputs=inputs,   # PROCESS_ID is the int, e.g. 89
                        queue=QUEUE, dedup=True, tag=TAG)
    body = r.json() if r.status_code == 202 else {}
    job_id, status = body.get("jobID") or body.get("id"), body.get("status", r.text)
    print(f"[{i}/{len(SUBMIT)}] {r.status_code} job_id={job_id} status={status}")
    rows.append({"n": i, "item_id": gr["item_id"], "job_id": job_id,
                 "submit_status": status, "http": r.status_code,
                 "submit_time": datetime.datetime.now().isoformat()})

submit_df = pd.DataFrame(rows)
out_dir = os.path.expanduser("~/my-public-bucket/dps_submission_results")
os.makedirs(out_dir, exist_ok=True)
stamp = datetime.datetime.now().strftime("%Y%m%d%H%M")
csv_path = f"{out_dir}/biomass-e2e_{TAG}_{stamp}.csv"
submit_df.to_csv(csv_path, index=False)
print("saved", csv_path)
submit_df

## 6. Monitor jobs and fetch results

`get_job_status` / `get_job_result` return JSON in maap-py v5. Results land under
`~/my-private-bucket/dps_output/rift-biomass-e2e/...` — the amplitude/phase/intensity COGs
plus `gate_prob.tif` + `unet_prob.tif`.

In [ ]:
for job_id in [j for j in submit_df["job_id"].tolist() if j]:
    s = maap.get_job_status(job_id)
    st = s.json().get("status") if s.status_code == 200 else s.text
    print(job_id, "->", st)

In [ ]:
# Once a job shows succeeded, inspect its outputs:
SUCCESS_JOB_ID = ""  # paste a job id
if SUCCESS_JOB_ID:
    r = maap.get_job_result(SUCCESS_JOB_ID)
    print(json.dumps(r.json(), indent=2) if r.status_code == 200 else r.text)
    m = maap.get_job_metrics(SUCCESS_JOB_ID)
    if m.status_code == 200:
        print(json.dumps(m.json(), indent=2))